In [ ]:
monthly_behavior = spark.sql("""
with monthly_stats as (
    select 
        month(hour_ts) as month_num,
        date_format(hour_ts, 'MMMM') as month_name,
        stddev(percent_change) as fluctuation,
        round(mean(percent_change), 2) as mean
    from `electric_analysis_dev`.`transformer_hourly_analysis`
    where load_change = 'Increase'
    group by 1, 2
),
trends as (
    select 
        month_name,
        mean,
        round(fluctuation, 2) as fluctuation
    from monthly_stats
)
select * from trends
order by fluctuation desc
""")

spark.sql("create database if not exists `electric_analysis_dev`")


monthly_behavior.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://ops-autopilot-data/transformed/monthly_behavior/") \
    .saveAsTable("`electric_analysis_dev`.`monthly_behavior`")